# M4 — Sistema multimodal (Turno 2) — validación en Colab
### SI4006 · Proyecto Integrador · Universidad EAFIT

Este notebook corre `m4/pipeline/sistema_multimodal.py` con GPU. El sistema completo
ahora carga **cuatro modelos**: el cerebro del agente (Qwen2.5-3B), el generador del
RAG (Qwen2.5-1.5B), BETO+LoRA (M1), y el clasificador visual de Camilo (ViT +
regresión logística, M4) — no cabe en una laptop sin GPU.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU
(T4)`.

Qué hace este notebook, en orden:
1. Clona el repo (rama `main`) e instala dependencias.
2. Corre los tests de lógica pura (disponibilidad condicional de `ver_imagen`, síntesis
   final con foto no evaluable) — sin GPU.
3. Prueba `ver_imagen` sola sobre una foto real del eval set de Camilo.
4. Corre `sistema_multimodal` **sin foto** y confirma que `ver_imagen` NO se usó.
5. Corre `sistema_multimodal` **con foto** y confirma que `ver_imagen` SÍ se usó.
6. Mide la latencia de punta a punta de los dos casos.
7. Corre los 10 casos del eval set con imagen y mide `ver()` por tono de piel (`m4/eval_set/`, Turno 3).

## 1 · Preparar el entorno

In [ ]:
import os

if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout main
    !git pull origin main
else:
    !git clone --branch main https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia

!git log -1 --oneline  # confirmar a simple vista qué commit quedó activo

In [ ]:
%pip install -q transformers datasets peft accelerate pandas torch scikit-learn sentence-transformers chromadb rank-bm25 pyyaml pillow

# Mismo arreglo que en los notebooks anteriores de M2/M3: Colab trae una version
# vieja de torchao preinstalada que hace tronar a `peft` al importarse.
!pip uninstall -y -q torchao

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU activa. Ve a Entorno de ejecución -> Cambiar tipo de "
          "entorno de ejecución -> T4 GPU, y vuelve a correr esta celda.")

## 2 · Tests de lógica pura (sin modelo, sin GPU)

Ya se corrieron en local durante el desarrollo; se repiten aquí para dejar registro de
que el código que se va a ejecutar con los modelos reales pasa estas pruebas: que
`ver_imagen` solo aparece disponible cuando hay foto, que pedirla sin foto da error, y
que la síntesis final nunca lee una foto "no evaluable" como si fuera "no urgente".

In [ ]:
import sys
sys.path.insert(0, "scripts")
import agente_tools as at

sin_foto = at._herramientas_disponibles(None)
con_foto = at._herramientas_disponibles("foto.jpg")
assert "ver_imagen" not in sin_foto and "ver_imagen" in con_foto

r = at._parsear_decision('{"accion": "usar_herramienta", "herramienta": "ver_imagen"}', sin_foto)
assert r["accion"] == "error", r

r = at._sintetizar_respuesta_final({
    "clasificador_beto": {"herramienta": "clasificador_beto", "etiqueta": "no_urgente", "confianza": 0.9},
    "ver_imagen": {"herramienta": "ver_imagen", "etiqueta": None, "score": None, "no_evaluable": True,
        "motivo_no_evaluable": "imagen desenfocada"},
})
assert "NO equivale" in r and "'no_urgente'" in r

print("Todos los tests de lógica pura pasaron.")

## 3 · Probar `ver_imagen` sola

Sobre una foto real del eval set que armó Camilo (`m4/pipeline/eval_tool/imagenes/`).

In [ ]:
import os

_imagenes = sorted(os.listdir("m4/pipeline/eval_tool/imagenes"))
_ruta_ejemplo = os.path.join("m4/pipeline/eval_tool/imagenes", _imagenes[0])
print("Imagen de prueba:", _ruta_ejemplo)

r_ver = at.tool_ver_imagen("", imagen=_ruta_ejemplo)
print(r_ver)

## 4 · `sistema_multimodal` — sin foto (debe confirmar que NO usa `ver_imagen`)

In [ ]:
sys.path.insert(0, "m4/pipeline")
from sistema_multimodal import sistema_multimodal
import time

pregunta_sin_foto = "Tengo un lunar que me cambió de color y de forma en los últimos meses."

t0 = time.perf_counter()
resultado_sin_foto = sistema_multimodal(pregunta_sin_foto)
latencia_sin_foto = time.perf_counter() - t0

print("Tools usadas:", resultado_sin_foto["tools_usadas"])
assert "ver_imagen" not in resultado_sin_foto["tools_usadas"], "ver_imagen no debería usarse sin foto"
print("OK: ver_imagen NO se usó (correcto, no había foto)")
print("Respuesta:", resultado_sin_foto["respuesta"][:300])
print(f"Latencia end-to-end (sin foto): {latencia_sin_foto:.2f} s")

## 5 · `sistema_multimodal` — con foto (debe confirmar que SÍ usa `ver_imagen`)

In [ ]:
pregunta_con_foto = "Le adjunto una foto de la lesión, ¿debería verla pronto?"

t0 = time.perf_counter()
resultado_con_foto = sistema_multimodal(pregunta_con_foto, imagen=_ruta_ejemplo)
latencia_con_foto = time.perf_counter() - t0

print("Tools usadas:", resultado_con_foto["tools_usadas"])
assert "ver_imagen" in resultado_con_foto["tools_usadas"], "ver_imagen debería haberse usado, había foto"
print("OK: ver_imagen SÍ se usó (correcto, había foto)")
print("Respuesta:", resultado_con_foto["respuesta"][:300])
print(f"Latencia end-to-end (con foto): {latencia_con_foto:.2f} s")

print()
print(f"Diferencia de latencia por agregar la foto: {latencia_con_foto - latencia_sin_foto:.2f} s")

## 6 · Guardar resultados

Para dejar evidencia real (no solo el código) de que el enchufe funciona, y que quede
disponible para la sección 3 del README y para quien siga en el relevo de M4.

In [ ]:
import json as _json

resumen = {
    "imagen_usada": _ruta_ejemplo,
    "sin_foto": {
        "pregunta": pregunta_sin_foto,
        "tools_usadas": resultado_sin_foto["tools_usadas"],
        "respuesta": resultado_sin_foto["respuesta"],
        "latencia_s": round(latencia_sin_foto, 4),
    },
    "con_foto": {
        "pregunta": pregunta_con_foto,
        "tools_usadas": resultado_con_foto["tools_usadas"],
        "respuesta": resultado_con_foto["respuesta"],
        "detalle_ver_imagen": resultado_con_foto["detalle_tools"].get("ver_imagen"),
        "latencia_s": round(latencia_con_foto, 4),
    },
}
with open("resultados_sistema_multimodal.json", "w", encoding="utf-8") as f:
    _json.dump(resumen, f, ensure_ascii=False, indent=2)
print("Guardado: resultados_sistema_multimodal.json")
resumen

In [ ]:
from google.colab import files
files.download("resultados_sistema_multimodal.json")

## 7 · Eval set con imagen (Turno 3)

Corre una vez los 10 casos de `m4/eval_set/casos_imagen.json` por `sistema_multimodal` y
deja `m4/eval_set/primera_corrida.json`: respuesta, tools usadas, lo que dijo cada tool,
veredicto según la regla del equipo y latencia. Es una revisión preliminar; la
calificación oficial la hace `calificar()` del harness (Turno 4).

img_09 e img_10 (piel oscura) ya están en el repo; `piel_oscura.py casos` solo hace falta
para volver a bajarlas del ISIC Archive.

In [ ]:
!python m4/eval_set/correr_casos_imagen.py

In [ ]:
# Sesgo por tono de piel: ver() sobre ~190 imágenes del ISIC Archive (necesita internet).
# Deja m4/eval_set/piel_oscura/resultados_piel_oscura.csv y resumen_piel_oscura.json.
!python m4/eval_set/piel_oscura/piel_oscura.py medir

---

Con esto queda validado el enchufe completo de M4: `ver_imagen` solo se ofrece cuando
hay foto (confirmado con y sin foto), `sistema_multimodal` corre de punta a punta de
forma reproducible desde el repo (no un mock), y queda medida la latencia de ambos
casos para la sección 3 del README.